# Epic 8 -- Dataset Export

After generating synthetic data with the unified pipeline, you often need to
convert it to a format that downstream training frameworks expect.

Epic 8 supports three export targets:

| Format | Use Case |
|--------|----------|
| **COCO** | Detectron2, MMDetection, torchvision |
| **YOLO** | Ultralytics YOLOv8/v11 |
| **HuggingFace** | `datasets.load_dataset("imagefolder")` |

## Setup: Generate a Small Dataset

In [ ]:
import tempfile
from pathlib import Path
import numpy as np
from PIL import Image

# Create a tiny test dataset
data_dir = Path(tempfile.mkdtemp(prefix="epic8_export_demo_"))
(data_dir / "images").mkdir()
(data_dir / "masks").mkdir()

for i in range(10):
    img = np.random.randint(50, 200, (128, 128, 3), dtype=np.uint8)
    mask = np.zeros((128, 128), dtype=np.uint8)
    mask[30:60, 40:90] = 255  # synthetic defect region
    Image.fromarray(img).save(str(data_dir / "images" / f"sample_{i:03d}.png"))
    Image.fromarray(mask).save(str(data_dir / "masks" / f"sample_{i:03d}_mask.png"))

print(f"Dataset created at: {data_dir}")

## COCO Export

In [ ]:
from udm_epic8.export.dataset_export import export_to_coco
import json

coco_path = export_to_coco(
    str(data_dir),
    str(data_dir / "coco_annotations.json"),
    modality="xray",
)

with open(coco_path) as f:
    coco = json.load(f)

print(f"Images:      {len(coco['images'])}")
print(f"Annotations: {len(coco['annotations'])}")
print(f"Categories:  {coco['categories']}")
print(f"\nFirst annotation: {json.dumps(coco['annotations'][0], indent=2)}")

## YOLO Export

In [ ]:
from udm_epic8.export.dataset_export import export_to_yolo

yolo_dir = export_to_yolo(str(data_dir), str(data_dir / "yolo_out"))
print(f"YOLO output: {yolo_dir}")
print(f"Labels: {list((yolo_dir / 'labels').glob('*.txt'))[:3]}")

# Show first label
first_label = sorted((yolo_dir / "labels").glob("*.txt"))[0]
print(f"\nFirst label ({first_label.name}):")
print(first_label.read_text())

## HuggingFace Export

In [ ]:
from udm_epic8.export.dataset_export import export_to_hf

hf_dir = export_to_hf(str(data_dir), str(data_dir / "hf_out"))
print(f"HF output: {hf_dir}")

# Show metadata
metadata = (hf_dir / "metadata.jsonl").read_text().strip().split("\n")
print(f"Metadata entries: {len(metadata)}")
print(f"First entry: {metadata[0]}")

## Merging Datasets

In [ ]:
from udm_epic8.export.dataset_export import merge_datasets

# Create a second small dataset to merge with
data_dir2 = Path(tempfile.mkdtemp(prefix="epic8_export_demo2_"))
(data_dir2 / "images").mkdir()
(data_dir2 / "masks").mkdir()
for i in range(5):
    img = np.random.randint(50, 200, (128, 128, 3), dtype=np.uint8)
    Image.fromarray(img).save(str(data_dir2 / "images" / f"aoi_{i:03d}.png"))

merged_manifest = merge_datasets(
    [str(data_dir), str(data_dir2)],
    str(data_dir / "merged"),
)

with open(merged_manifest) as f:
    merged = json.load(f)
print(f"Total merged images: {merged['total_images']}")
print(f"Sources: {merged['sources']}")

## CLI Usage

All export operations are available via the CLI:

```bash
# Export to COCO
udm-epic8 export --data-dir data/universal/xray --format coco --output exports/coco.json

# Export to YOLO
udm-epic8 export --data-dir data/universal/xray --format yolo --output exports/yolo

# Export to HuggingFace
udm-epic8 export --data-dir data/universal/xray --format hf --output exports/hf

# Merge datasets
udm-epic8 merge --dirs data/universal/xray --dirs data/universal/aoi --output data/merged
```